# Chapter 18: Stateful and Multi-Turn Testing

**Learning objectives:**
- Test agent memory retention across turns using `StatefulAgentTester`
- Detect state degradation via geodesic and tension metrics
- Use `MultiTurnAdapter` and `InteractionScript` for scripted interaction sequences
- Measure cross-turn consistency via `tension_energy()`

---
## Preamble

Chapters 16--17 tested single-turn behavioral properties: hallucination and calibration. This chapter addresses **stateful testing** — verifying that agents maintain memory across conversation turns and degrade gracefully under sustained interaction.

Two complementary subsystems:

1. **StatefulAgentTester**: teach facts, insert distractors, test recall. Uses `score_triple()` geodesic for recall quality and `tension_energy()` for session drift.

2. **MultiTurnAdapter**: execute scripted `InteractionScript` objects with per-turn assertions on output content, tool calls, latency and environment state.

In [ ]:
# === Preamble (all notebooks) ===
import warnings
warnings.filterwarnings("ignore")

import torch
import numpy as np

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

In [ ]:
# [AUTO-INJECTED] Disable DOE tracing for parallel execution safety
from knowlytix.harness.testing import DOEHarnessConfig
DOEHarnessConfig.__dataclass_fields__['enable_tracing'].default = False


In [ ]:
from importlib.resources import files
# Load the Chapter 4 store
from knowlytix.harness.testing import DOEGMSBenchmark, DOEHarnessConfig

config = DOEHarnessConfig(
    markdown_path=str(files("knowlytix.harness.testing").joinpath("tests/model_risk_assessment.md")),
    stores_dir="../lab_stores",
    store_name="ch04_ground_truth",
    n_runs=4,
    ingest_epochs=200,
    device=str(DEVICE),
)
bench = DOEGMSBenchmark(config)
store = bench.ingest()
print(f"Store: {store.stats()['entities']} entities, "
      f"{store.stats()['triples']} triples, "
      f"{store.stats()['enm_entries']} ENM")

---

## Stateful Agent Testing: Teach, Delay, Recall

The `StatefulAgentTester` mines triples from the GMS store and constructs memory probes:

| Class | Fields | Purpose |
|---|---|---|
| `AgentMemoryProbe` | probe_id, fact_triple, teach_prompt, recall_prompt, delay_turns, expected_answer, recall_geodesic, session_tension | Single memory test |
| `MemoryAssessment` | n_probes, n_recalled, mean_recall_geodesic, mean_session_tension, degradation_detected, probes | Aggregate results |
| `MemoryRecallOracle` | store, geodesic_threshold | Scores recall via `score_triple()` |
| `StatefulAgentTester` | store | Builds probes, scores recall, produces assessments |

In [ ]:
# Build memory probes from the GMS store
from knowlytix.harness.testing.stateful import (
    StatefulAgentTester, MemoryRecallOracle,
)

tester = StatefulAgentTester(store=store)
probes = tester.build_probes(n_probes=5, delay_turns=3, seed=42)

print(f"Built {len(probes)} memory probes (delay={probes[0].delay_turns} turns)")
print(f"\n{'Probe':<16s} {'Fact Triple':<55s} {'Expected'}")
print("-" * 95)
for p in probes:
    h, r, t = p.fact_triple
    triple_str = f"({h}, {r}, {t})"
    print(f"{p.probe_id:<16s} {triple_str:<55s} {p.expected_answer}")

In [ ]:
# Demonstrate teach/recall on FullMockHarness
from knowlytix.harness.fixtures import FullMockHarness

harness = FullMockHarness(recovery_mode="fallback")

# Teach a fact
harness.teach("project_deadline", "2026-06-15")
harness.teach("model_auc", "0.8412")

# Distractor turns
_ = harness.run("What is the test AUC of MDL-001?")
_ = harness.run("What domain is MDL-002 in?")

# Recall
r1 = harness.recall("project_deadline")
r2 = harness.recall("model_auc")
r3 = harness.recall("nonexistent_key")

print("=== Teach/Recall Demo ===")
print(f"  project_deadline: {r1!r}")
print(f"  model_auc:        {r2!r}")
print(f"  nonexistent_key:  {r3!r}")

---

## MemoryRecallOracle: Scoring Recall Quality

The `MemoryRecallOracle` scores a recall attempt by comparing the expected and recalled triples via `score_triple()` geodesic distance and `tension_energy()` between them.

In [6]:
# MemoryRecallOracle: compare expected vs recalled
oracle = MemoryRecallOracle(store=store, geodesic_threshold=0.5)

# Perfect recall
result_good = oracle.evaluate(
    head="MDL-001", relation="has_dependency_type",
    expected_tail="Credit scores segment LGD pools",
    recalled_answer="Credit scores segment LGD pools",
)
print("=== Perfect Recall ===")
for k, v in result_good.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.4f}")
    else:
        print(f"  {k}: {v}")

# Wrong recall
result_bad = oracle.evaluate(
    head="MDL-001", relation="has_dependency_type",
    expected_tail="Credit scores segment LGD pools",
    recalled_answer="0.8934",
)
print("\n=== Wrong Recall ===")
for k, v in result_bad.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.4f}")
    else:
        print(f"  {k}: {v}")

=== Perfect Recall ===
  oracle_type: memory_recall
  passed: False
  geodesic_expected: 1.2941
  geodesic_recalled: 1.2941
  tension_expected_vs_recalled: 0.0001
  expected_tail: Credit scores segment LGD pools
  recalled_answer: Credit scores segment LGD pools



=== Wrong Recall ===
  oracle_type: memory_recall
  passed: False
  geodesic_expected: 1.2941
  geodesic_recalled: 1.5487
  tension_expected_vs_recalled: 1.3652
  expected_tail: Credit scores segment LGD pools
  recalled_answer: 0.8934


---

## Lab 18.1 -- Test Agent Memory Across Turns

In this lab we run the full teach-delay-recall cycle using the `StatefulAgentTester` and produce a `MemoryAssessment`.

In [ ]:
from importlib.resources import files
# Lab 18.1a -- Run memory probes through FullMockHarness
harness = FullMockHarness(recovery_mode="fallback")
context = open(str(files("knowlytix.harness.testing").joinpath("tests/model_risk_assessment.md"))).read()

tester = StatefulAgentTester(store=store)
probes = tester.build_probes(n_probes=10, delay_turns=3, seed=42)

for p in probes:
    # Teach phase
    harness.run(p.teach_prompt)
    # Distractor turns
    for _ in range(p.delay_turns):
        harness.run("What is the test AUC of MDL-001?")
    # Recall phase
    answer = harness.run(p.recall_prompt)
    tester.score_recall(p, answer)

assessment = tester.assess(probes)
print(f"=== MemoryAssessment ({assessment.n_probes} probes) ===")
print(f"  Recalled:              {assessment.n_recalled}/{assessment.n_probes}")
print(f"  Mean recall geodesic:  {assessment.mean_recall_geodesic:.4f}")
print(f"  Mean session tension:  {assessment.mean_session_tension:.4f}")
print(f"  Degradation detected:  {assessment.degradation_detected}")

In [8]:
# Lab 18.1b -- Per-probe detail
print(f"{'Probe':<16s} {'Expected':<30s} {'Geo':>8s} {'Tension':>8s}")
print("-" * 68)
for p in probes:
    geo = f"{p.recall_geodesic:.4f}" if p.recall_geodesic is not None else "None"
    ten = f"{p.session_tension:.4f}" if p.session_tension is not None else "None"
    print(f"{p.probe_id:<16s} {p.expected_answer:<30s} {geo:>8s} {ten:>8s}")

Probe            Expected                            Geo  Tension
--------------------------------------------------------------------
mem_probe_000    Stable → Degraded                1.5603     None
mem_probe_001    Quarterly                        1.5043     None
mem_probe_002    FraudNet-LSTM                    1.3285     None
mem_probe_003    FND-007                            None     None
mem_probe_004    0.8489                             None     None
mem_probe_005    0.7812                           1.6045     None
mem_probe_006    0.8567                             None     None
mem_probe_007    2,300,000                          None     None
mem_probe_008    AML Team                           None     None
mem_probe_009    Semi-Annual                      1.5722     None


In [9]:
# Lab 18.1c -- Vary delay turns and measure degradation
delays = [1, 3, 5, 10]
delay_results = []

for delay in delays:
    h = FullMockHarness(recovery_mode="fallback")
    t = StatefulAgentTester(store=store)
    ps = t.build_probes(n_probes=5, delay_turns=delay, seed=42)
    for p in ps:
        h.run(p.teach_prompt)
        for _ in range(p.delay_turns):
            h.run("What is the test AUC of MDL-001?")
        answer = h.run(p.recall_prompt)
        t.score_recall(p, answer)
    a = t.assess(ps)
    delay_results.append((delay, a))

print(f"{'Delay':>6s} {'Recalled':>10s} {'Mean Geo':>10s} {'Degraded':>10s}")
print("-" * 40)
for delay, a in delay_results:
    print(f"{delay:>6d} {a.n_recalled:>10d} {a.mean_recall_geodesic:>10.4f} "
          f"{str(a.degradation_detected):>10s}")

 Delay   Recalled   Mean Geo   Degraded
----------------------------------------
     1          0     1.4644      False
     3          0     1.4644      False
     5          0     1.4644      False
    10          0     1.4644      False


---

## Multi-Turn Testing with InteractionScript

The `InteractionScript` defines a scripted conversation with per-turn assertions:

| Class | Key Fields | Purpose |
|---|---|---|
| `TurnSpec` | turn_id, user_message, expected_contains, expected_not_contains, expected_tool_calls, prohibited_tool_calls, max_latency_ms, carry_forward | Single turn specification |
| `InteractionScript` | script_id, name, turns, setup_context, teardown_assertions | Complete multi-turn scenario |
| `TurnResult` | turn_id, raw_output, tool_calls, assertions_passed, assertions_failed | Per-turn outcome |
| `ScriptResult` | overall_passed, turn_results, consistency_scores, teardown_passed/failed | Aggregate outcome |

In [ ]:
# The pre-built model review script
from knowlytix.harness.testing.multiturn import (
    InteractionScript, TurnSpec, MultiTurnAdapter,
)
from knowlytix.harness.fixtures import create_multi_turn_script, create_mock_environment

script = create_multi_turn_script()
print(f"Script: {script.script_id}")
print(f"Name:   {script.name}")
print(f"Turns:  {len(script.turns)}")
for t in script.turns:
    print(f"\n  [{t.turn_id}]")
    print(f"    message:  {t.user_message[:60]}...")
    print(f"    expects:  {t.expected_contains}")
    print(f"    tools:    {t.expected_tool_calls}")

In [11]:
# Execute the pre-built script
env = create_mock_environment()
harness = FullMockHarness(environment=env, recovery_mode="fallback")
adapter = MultiTurnAdapter(harness=harness, store=store)

result = adapter.execute_script(script, environment=env)

print("=== ScriptResult ===")
print(f"  overall_passed: {result.overall_passed}")
print(f"  total_latency:  {result.total_latency_ms:.2f} ms")

for tr in result.turn_results:
    status = "PASS" if tr.passed else "FAIL"
    print(f"\n  [{tr.turn_id}] {status}")
    print(f"    output: {tr.raw_output[:60]!r}")
    if tr.assertions_passed:
        for a in tr.assertions_passed:
            print(f"    + {a}")
    if tr.assertions_failed:
        for a in tr.assertions_failed:
            print(f"    - {a}")

print(f"\n  Consistency: {result.consistency_scores}")
print(f"  Teardown passed: {result.teardown_passed}")
print(f"  Teardown failed: {result.teardown_failed}")

=== ScriptResult ===
  overall_passed: False
  total_latency:  1.22 ms

  [turn_1] FAIL
    output: '0.8934'
    + tool 'data_lookup' was called
    - missing expected '0.8412'

  [turn_2] PASS
    output: 'High'
    + contains 'High'
    + tool 'risk_check' was called

  [turn_3] FAIL
    output: "{'name': 'CreditScore-XGB', 'domain': 'Credit Decisioning', "
    - expected tool 'create_ticket' not called

  Consistency: {'0.8934@turn_0->turn_2': 0.00048828125, 'High@turn_1->turn_2': 9.999999747378752e-05}
  Teardown passed: []
  Teardown failed: ['teardown: only 1 turns passed, need 2']


---

## Lab 18.2 -- Build a Custom Interaction Script

In this lab we construct a custom 3-turn teach-distract-recall script and execute it through the `MultiTurnAdapter`.

In [ ]:
from importlib.resources import files
# Lab 18.2a -- Build a custom script
ctx = open(str(files("knowlytix.harness.testing").joinpath("tests/model_risk_assessment.md"))).read()

custom_script = InteractionScript(
    script_id="memory_test_001",
    name="Teach-Distract-Recall",
    description="Test agent memory across 3 turns",
    setup_context=ctx,
    turns=[
        TurnSpec(
            turn_id="teach",
            user_message="Remember: MDL-001 has a test AUC of 0.8412. Acknowledge this.",
            expected_contains=["MDL-001"],
        ),
        TurnSpec(
            turn_id="distract",
            user_message="What domain does MDL-002 operate in?",
            expected_not_contains=[],
        ),
        TurnSpec(
            turn_id="recall",
            user_message="What was the test AUC of MDL-001 that I told you earlier?",
            expected_contains=["0.8412"],
        ),
    ],
    teardown_assertions=[
        {"type": "all_turns_passed"},
    ],
)

print(f"Script: {custom_script.script_id} ({custom_script.name})")
print(f"Turns:  {len(custom_script.turns)}")
for t in custom_script.turns:
    print(f"  {t.turn_id}: expects={t.expected_contains or '(none)'}")

In [13]:
# Lab 18.2b -- Execute the custom script
harness = FullMockHarness(recovery_mode="fallback")
adapter = MultiTurnAdapter(harness=harness, store=store)

result_custom = adapter.execute_script(custom_script)

print("=== Custom Script Result ===")
print(f"  overall_passed: {result_custom.overall_passed}")

for tr in result_custom.turn_results:
    status = "PASS" if tr.passed else "FAIL"
    print(f"\n  [{tr.turn_id}] {status}")
    print(f"    output: {tr.raw_output[:60]!r}")
    if tr.assertions_passed:
        for a in tr.assertions_passed:
            print(f"    + {a}")
    if tr.assertions_failed:
        for a in tr.assertions_failed:
            print(f"    - {a}")

print(f"\n  Consistency: {result_custom.consistency_scores}")
print(f"  Teardown passed: {result_custom.teardown_passed}")
print(f"  Teardown failed: {result_custom.teardown_failed}")

=== Custom Script Result ===
  overall_passed: False

  [teach] FAIL
    output: '0.8934'
    - missing expected 'MDL-001'

  [distract] PASS
    output: 'Fraud Detection'

  [recall] FAIL
    output: '0.8934'
    - missing expected '0.8412'

  Consistency: {'0.8934@turn_0->turn_2': 0.00048828125}
  Teardown passed: []
  Teardown failed: ['teardown: 2 turn(s) failed']


In [14]:
# Lab 18.2c -- Tool-call focused script with environment
env = create_mock_environment()

tool_script = InteractionScript(
    script_id="tool_test_001",
    name="Tool Sequence Verification",
    description="Verify correct tool usage across turns",
    setup_context=ctx,
    turns=[
        TurnSpec(
            turn_id="lookup",
            user_message="Look up model MDL-001 and tell me about it.",
            expected_tool_calls=["data_lookup"],
        ),
        TurnSpec(
            turn_id="check_risk",
            user_message="What is the risk rating of MDL-001?",
            expected_tool_calls=["risk_check"],
        ),
        TurnSpec(
            turn_id="notify",
            user_message="Send a notification about MDL-001 findings.",
            expected_tool_calls=["send_notification"],
            prohibited_tool_calls=["dangerous_delete"],
        ),
    ],
    teardown_assertions=[
        {"type": "min_turns_passed", "value": 2},
    ],
)

harness_env = FullMockHarness(environment=env, recovery_mode="fallback")
adapter_env = MultiTurnAdapter(harness=harness_env, store=store)
result_tool = adapter_env.execute_script(tool_script, environment=env)

print("=== Tool Script Result ===")
print(f"  overall_passed: {result_tool.overall_passed}")
for tr in result_tool.turn_results:
    status = "PASS" if tr.passed else "FAIL"
    tools = [tc.get('tool_name', '?') if isinstance(tc, dict) else tc
             for tc in tr.tool_calls]
    print(f"  [{tr.turn_id}] {status}: tools={tools}")
    if tr.assertions_passed:
        for a in tr.assertions_passed:
            print(f"    + {a}")
    if tr.assertions_failed:
        for a in tr.assertions_failed:
            print(f"    - {a}")
print(f"  Teardown passed: {result_tool.teardown_passed}")
print(f"  Teardown failed: {result_tool.teardown_failed}")

=== Tool Script Result ===
  overall_passed: False
  [lookup] PASS: tools=['data_lookup', 'risk_check']
    + tool 'data_lookup' was called
  [check_risk] PASS: tools=['data_lookup', 'risk_check']
    + tool 'risk_check' was called
  [notify] FAIL: tools=['data_lookup', 'risk_check']
    + prohibited tool 'dangerous_delete' not called
    - expected tool 'send_notification' not called
  Teardown passed: ['teardown: 2 turns passed >= 2']
  Teardown failed: []


---

## End-to-End: Memory Probes at Scale

We run memory probes at scale to measure the aggregate recall quality across the full store.

In [15]:
# E2E: 20 memory probes with varying delays
harness_e2e = FullMockHarness(recovery_mode="fallback")
tester_e2e = StatefulAgentTester(store=store)
probes_e2e = tester_e2e.build_probes(n_probes=20, delay_turns=5, seed=42)

for p in probes_e2e:
    harness_e2e.run(p.teach_prompt)
    for _ in range(p.delay_turns):
        harness_e2e.run("Distractor question about model risk.")
    answer = harness_e2e.run(p.recall_prompt)
    tester_e2e.score_recall(p, answer)

assessment_e2e = tester_e2e.assess(probes_e2e)
print(f"=== E2E Memory Assessment ({assessment_e2e.n_probes} probes) ===")
print(f"  Recalled:              {assessment_e2e.n_recalled}/{assessment_e2e.n_probes}")
print(f"  Mean recall geodesic:  {assessment_e2e.mean_recall_geodesic:.4f}")
print(f"  Mean session tension:  {assessment_e2e.mean_session_tension:.4f}")
print(f"  Degradation detected:  {assessment_e2e.degradation_detected}")

=== E2E Memory Assessment (20 probes) ===
  Recalled:              0/20
  Mean recall geodesic:  1.4878
  Mean session tension:  0.0000
  Degradation detected:  False


In [16]:
# Geodesic distribution for recalled probes
geos = [p.recall_geodesic for p in probes_e2e if p.recall_geodesic is not None]
tensions = [p.session_tension for p in probes_e2e if p.session_tension is not None]

print(f"=== Recall Geodesic Distribution ({len(geos)} probes with scores) ===")
if geos:
    print(f"  Mean:   {np.mean(geos):.4f}")
    print(f"  Std:    {np.std(geos):.4f}")
    print(f"  Min:    {np.min(geos):.4f}")
    print(f"  Max:    {np.max(geos):.4f}")
else:
    print("  No geodesic scores available")

print(f"\n=== Session Tension Distribution ({len(tensions)} probes with scores) ===")
if tensions:
    print(f"  Mean:   {np.mean(tensions):.4f}")
    print(f"  Std:    {np.std(tensions):.4f}")
    print(f"  Min:    {np.min(tensions):.4f}")
    print(f"  Max:    {np.max(tensions):.4f}")
else:
    print("  No tension scores (mock agent does not produce entity names)")

# Summary table
print("\n=== Summary ===")
print(f"  {assessment_e2e.to_dict()}")

=== Recall Geodesic Distribution (9 probes with scores) ===
  Mean:   1.4878
  Std:    0.1148
  Min:    1.3260
  Max:    1.6045

=== Session Tension Distribution (0 probes with scores) ===
  No tension scores (mock agent does not produce entity names)

=== Summary ===
  {'n_probes': 20, 'n_recalled': 0, 'recall_rate': 0.0, 'mean_recall_geodesic': 1.4878, 'mean_session_tension': 0.0, 'degradation_detected': False}


---

## Exercises

**Exercise 18.1** -- Build 10 memory probes with `delay_turns=0` (immediate recall) and compare the `MemoryAssessment` against `delay_turns=10`. Does the mock agent's recall quality degrade with more distractor turns?

**Exercise 18.2** -- Create a 5-turn `InteractionScript` that progressively builds a model risk report: (1) look up model, (2) check risk, (3) create ticket, (4) send notification, (5) verify state. Add `teardown_assertions` for `all_turns_passed` and `total_tool_calls_max=15`.

**Exercise 18.3** -- Use the `MemoryRecallOracle` to evaluate recall quality for 20 probes. Compute the pass rate at two thresholds: `geodesic_threshold=0.5` (strict) and `geodesic_threshold=1.5` (lenient). What fraction of recalls pass at each threshold?

---

## Summary

This chapter introduced **stateful and multi-turn testing** — verifying agent behavior across conversation turns:

1. **StatefulAgentTester** builds memory probes from GMS store triples: teach a fact, insert distractor turns, test recall. `score_triple()` measures recall quality and `tension_energy()` detects session drift.

2. **MemoryAssessment** aggregates probe results into a report: recall count, mean geodesic, mean tension and degradation flag.

3. **MemoryRecallOracle** provides structured evaluation of recall attempts with geodesic distance and tension between expected and recalled answers.

4. **InteractionScript** with **TurnSpec** defines scripted multi-turn scenarios with per-turn assertions on output content, tool calls, latency and environment state.

5. **MultiTurnAdapter** executes scripts and measures **cross-turn consistency** via `tension_energy()` between entities mentioned in successive turns.

6. **Teardown assertions** verify post-script invariants: all turns passed, minimum turns passed, total tool call budget.

In the next chapter we turn to **multi-agent consensus and cross-document consistency** — testing whether multiple agents with different roles reach consistent conclusions and detecting factual conflicts across document stores.